# g-and-k Simulator: General Score-Matching Method Notebook

This notebook demonstrates how to run the general (non-conjugate)
neural score-matching Bayes procedure for the g-and-k simulator.
It mirrors the experiment in `scripts/normflow/run_gnk.py`.

It covers:
1. Setting up the g-and-k simulator (`nsm_bayes.simulators`).
2. Generating training data and training a normalising-flow
   likelihood estimator `q_phi` (`nsm_bayes.models.general`).
3. Generating observed data with outliers
   (`nsm_bayes.simulators.add_noise`).
4. Sampling the NLE posterior under misspecification with MCMC.
5. Calibrating $\beta$ via the GPC method (`nsm_bayes.inference.general.calibrate`).
6. Sampling the score-matching posterior and comparing the
   posterior means to the true parameters.


In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from nsm_bayes.inference.general.calibrate import calibrate_beta
from nsm_bayes.method import ScoreMatchingLogPosterior
from nsm_bayes.models.general import make_nle_logprob, train_normflow
from nsm_bayes.models.general.normflow import run_mcmc
from nsm_bayes.simulators.add_noise import add_outliers_by_proportion
from nsm_bayes.simulators.benchmark_simulators.simulators import (
    GandKSimulator,
)
from nsm_bayes.inference.general.slice_sampler import run_multivariate_slice_sampler_tuned
from nsm_bayes.utils.statistics import robust_mean_cov
from nsm_bayes.utils.linalg import sample_mean_and_covariance, sigma_inverse


## 1. Configuration

The g-and-k model is parametrised as
$(A, \log B, g, \log k)$, so the Gaussian prior is defined
directly on this 4-dimensional vector and the observed data are
scalar ($d_x = 1$).

The values below follow the Hydra config `configs/gnk.yaml`
(with a smaller `num_samples` so the notebook runs quickly).


In [ ]:
cfg = {
    "prior_mean": [0.0, 0.7, 0.0, -1.5],
    "prior_cov": [
        [5.0, 0.0, 0.0, 0.0],
        [0.0, 0.5, 0.0, 0.0],
        [0.0, 0.0, 4.0, 0.0],
        [0.0, 0.0, 0.0, 0.25],
    ],
    "theta_true": [1.0, 0.5, 1.0, -1.0],
    "num_samples": 10000,      # training samples (100000 in configs/gnk.yaml)
    "n_obs": 100,              # number of observed data points
    "epsilon": 0.1,            # proportion of outliers
    "outlier_values": [-50.0], # location of the outliers
    "robust_flag": False,      # robust or classical mean/covariance
    "num_posterior_samples": 500,
    "num_chains": 1,
    "warmup_steps": 500,
    "thin": 1,
    "beta_base_general": 1.0,  # starting beta for the GPC calibration
    "alpha": 0.05,             # target 1 - alpha credible coverage
    "B": 100,                  # bootstrap samples per iteration
    "T": 20,                   # number of calibration iterations
}

random.seed(1)
np.random.seed(1)
torch.manual_seed(1)
dtype = torch.float32
d_x = 1


## 2. Simulator and prior

The simulator exposes a Gaussian prior in the reparameterised
space through `GandKSimulator.prior`.


In [ ]:
prior_mean = torch.tensor(cfg["prior_mean"], dtype=dtype)
prior_cov = torch.tensor([list(r) for r in cfg["prior_cov"]], dtype=dtype)
theta_true = torch.tensor(cfg["theta_true"], dtype=dtype)

sim = GandKSimulator(prior_mean=prior_mean, prior_cov=prior_cov)
prior = sim.prior()

print(f"Prior mean: {prior_mean}")
print(f"True theta: {theta_true}")


## 3. Training data and NLE likelihood estimator

Simulate $(\theta, x)$ pairs from the prior and train a
normalising-flow likelihood estimator $q_\phi(x \mid \theta)$
with `train_normflow` (sbi `SNLE` under the hood).


In [ ]:
print("Simulating training data...")
theta = prior.sample((cfg["num_samples"],))
x_sim = sim.simulate(theta).unsqueeze(-1)
print(f"theta: {tuple(theta.shape)}, x_sim: {tuple(x_sim.shape)}")

print("Training the normalising-flow likelihood estimator...")
start_time = time.perf_counter()
likelihood_estimator = train_normflow(theta, x_sim, prior)
cost_nle_training = time.perf_counter() - start_time
print(f"NLE training took {cost_nle_training:.1f} s")


## 4. Observed data with outliers

Draw `n_obs` scalar observations from the model at `theta_true`
and contaminate a proportion `epsilon` of them with outliers at
`outlier_values`, mimicking model misspecification.


In [ ]:
x_obs = sim.simulate(theta_true, n=cfg["n_obs"]).unsqueeze(-1)
x_obs_mis, outlier_indices = add_outliers_by_proportion(
    x_obs, cfg["epsilon"], cfg["outlier_values"]
)
print(
    f"Generated {cfg['n_obs']} observations with "
    f"{len(outlier_indices)} outliers."
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, data, title in [
    (axes[0], x_obs, "Clean data"),
    (axes[1], x_obs_mis, "Contaminated data"),
]:
    ax.hist(data.detach().cpu().numpy(), bins=30, edgecolor="black")
    ax.set_title(title)
    ax.set_xlabel("x")
    axes[0].set_ylabel("Count")
plt.tight_layout()
plt.show()


## 5. NLE posterior (baseline under misspecification)

Sample the standard neural-likelihood posterior
$p(\theta \mid x_{\mathrm{mis}})$ with MCMC (`run_mcmc`). This
baseline is sensitive to the outliers.


In [ ]:
print("Sampling the NLE posterior with MCMC...")
start_time = time.perf_counter()
samples_nle_mis = run_mcmc(
    x_obs_mis,
    likelihood_estimator,
    num_pos_samples=cfg["num_posterior_samples"],
    num_chains=cfg["num_chains"],
    num_workers=cfg["num_chains"],
    thin=cfg["thin"],
    warmup_steps=cfg["warmup_steps"],
)
cost_nle_mcmc = time.perf_counter() - start_time
print(f"NLE MCMC took {cost_nle_mcmc:.1f} s")
print(f"samples_nle_mis: {tuple(samples_nle_mis.shape)}")


## 6. Score-matching posterior

The general score-matching log-posterior is

$$\log p_{\mathrm{sm}}(\theta) = \log p(\theta) - \beta\, n\,
L_{\mathrm{sm}}(\theta, x_{\mathrm{mis}}),$$

with inverse-multiquadric (IMQ) weights centred at `mu_hat`
with inverse covariance `Sigma_inv`.
Following `run_gnk.py`, the weight scale is fixed at `c = 1`.


In [ ]:
if cfg["robust_flag"]:
    mu_hat, Sigma_hat = robust_mean_cov(x_obs_mis)
else:
    mu_hat, Sigma_hat = sample_mean_and_covariance(x_obs_mis)
Sigma_inv = sigma_inverse(Sigma_hat)
c = 1.0

print(f"mu_hat: {mu_hat}")
print(f"Sigma_hat: {Sigma_hat}")


In [ ]:
def make_sm_logposterior(beta: float) -> ScoreMatchingLogPosterior:
    """Build the score-matching log-posterior for a given beta."""
    return ScoreMatchingLogPosterior(
        x_obs=x_obs_mis,
        prior=prior,
        beta=beta,
        q_phi_log_prob=likelihood_estimator,
        mu_hat=mu_hat,
        Sigma_inv=Sigma_inv,
        c=c,
        weight_type="imq",
    )


## 7. Calibrating $\beta$ (GPC)

First run one slice-sampler pass at `beta_base_general` to obtain
base samples, then let `calibrate_beta` tune `beta` so that the
bootstrap coverage matches $1 - \alpha$. The `refresh_sampler`
callback redraws samples whenever the effective sample size drops
too low.


In [ ]:
print("Sampling base posterior at beta_base...")
start_time = time.perf_counter()
sm_lp_base = make_sm_logposterior(cfg["beta_base_general"])
theta_samples_base = run_multivariate_slice_sampler_tuned(
    log_posterior_fn=sm_lp_base,
    prior=prior,
    num_samples=cfg["num_posterior_samples"],
    num_chains=cfg["num_chains"],
    warmup_steps=cfg["warmup_steps"],
    thin=cfg["thin"],
)
theta_samples_base = torch.from_numpy(theta_samples_base).to(dtype)

def refresh_sampler_at(beta_new: float) -> torch.Tensor:
    """Redraw samples for a new beta during calibration."""
    sm_lp = make_sm_logposterior(beta_new)
    arr = run_multivariate_slice_sampler_tuned(
        log_posterior_fn=sm_lp,
        prior=prior,
        num_samples=cfg["num_posterior_samples"],
        num_chains=cfg["num_chains"],
        warmup_steps=cfg["warmup_steps"],
        thin=cfg["thin"],
    )
    return torch.from_numpy(arr).to(dtype)


In [ ]:
print("Calibrating beta with GPC...")
beta, history_general = calibrate_beta(
    theta_samples_base=theta_samples_base,
    beta_base=cfg["beta_base_general"],
    x_obs=x_obs_mis,
    q_phi_log_prob=make_nle_logprob(likelihood_estimator),
    mu_hat=mu_hat,
    Sigma_inv=Sigma_inv,
    weight_type="imq",
    c=c,
    alpha=cfg["alpha"],
    B=cfg["B"],
    T=cfg["T"],
    step_schedule=lambda t: 10.0 / (t + 10.0),
    refresh_sampler=refresh_sampler_at,
)
print(f"Calibrated beta: {beta:.6f}")


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history_general["betas"], marker="o", label="beta")
plt.axhline(cfg["alpha"], color="r", ls="--", label=r"$1 - \alpha$ target")
plt.xlabel("GPC iteration")
plt.ylabel("Beta / target coverage")
plt.title("GPC calibration of $\beta$")
plt.legend()
plt.grid(True)
plt.show()


## 8. Final score-matching posterior

Run the slice sampler one last time at the calibrated `beta`
and compare the posterior means of both methods to the true
parameters.


In [ ]:
print("Sampling the calibrated score-matching posterior...")
log_posterior_calculator = make_sm_logposterior(beta)
samples_sm_general = run_multivariate_slice_sampler_tuned(
    log_posterior_fn=log_posterior_calculator,
    prior=prior,
    num_samples=cfg["num_posterior_samples"],
    num_chains=cfg["num_chains"],
    warmup_steps=cfg["warmup_steps"],
    thin=cfg["thin"],
)
print(f"samples_sm_general: {tuple(samples_sm_general.shape)}")
mean_sm = torch.from_numpy(samples_sm_general.mean(axis=0)).to(dtype)
mean_nle = samples_nle_mis.mean(dim=0)

print("\nTrue theta:")
print(theta_true)
print("\nNLE posterior mean:")
print(mean_nle)
print("\nScore-matching posterior mean:")
print(mean_sm)


In [ ]:
labels = ["A", r"log B", "g", r"log k"]
std_sm = torch.from_numpy(
    samples_sm_general.std(axis=0, ddof=1)
).to(dtype)

x_pos = np.arange(len(labels))
plt.figure(figsize=(10, 5))
plt.errorbar(
    x_pos,
    mean_sm.detach().cpu().numpy(),
    yerr=std_sm.detach().cpu().numpy(),
    fmt="o",
    capsize=5,
    label="Score matching",
)
plt.scatter(
    x_pos, mean_nle.detach().cpu().numpy(),
    marker="s", color="gray", label="NLE (baseline)",
)
plt.plot(
    x_pos, theta_true.detach().cpu().numpy(),
    "rx--", marker="x", markersize=10, label="True",
)
plt.xticks(x_pos, labels)
plt.ylabel("Value")
plt.title("Posterior means under misspecification (g-and-k)")
plt.legend()
plt.grid(True, axis="y")
plt.show()


## Summary

- `theta_samples_base`: base MCMC samples at `beta_base_general`.
- `beta`: GPC-calibrated score-matching strength.
- `samples_sm_general`: posterior samples under the calibrated
  score-matching posterior.
- `samples_nle_mis`: baseline NLE posterior samples under the
  same contaminated data.

The score-matching posterior should stay closer to the true
parameters than the NLE posterior when the observations contain
outliers.
